# 🚀 Crypto Multi-Timeframe Training (Kaggle Optimized)

**Цель:** Обучение LSTM модели на 5 топовых криптовалютах и 7 таймфреймах.

**Важно:**
1. Включите **Internet** в настройках ноутбука (Settings -> Internet -> On).
2. Выберите ускоритель **GPU T4 x2** (Settings -> Accelerator -> GPU T4 x2).
3. Запустите `Run All`.

In [ ]:
# ==========================================
# 1. Установка зависимостей
# ==========================================
!pip install yfinance pandas numpy scikit-learn torch --quiet
print("✅ Зависимости установлены")

In [ ]:
# ==========================================
# 2. Импорт библиотек и проверка окружения
# ==========================================
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime

warnings.filterwarnings('ignore')

# Проверка GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU обнаружен: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("⚠️ GPU не найден, используем CPU (медленно)")

# Создание директорий
os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/data', exist_ok=True)
print("✅ Директории созданы")

In [ ]:
# ==========================================
# 3. Конфигурация
# ==========================================
SYMBOLS = ['BTC-USD', 'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD']
TIMEFRAMES_MAP = {
    '5m': '5m', '15m': '15m', '1h': '1h', 
    '4h': '1d', '12h': '1d', '1d': '1d', '1w': '1wk'
}
# Примечание: yfinance не поддерживает 4h и 12h напрямую, будем эмулировать ресемплингом из 1h/1d

LOOKBACK_PERIODS = {
    '5m': 60, '15m': 60, '1h': 48, 
    '4h': 50, '12h': 60, '1d': 60, '1w': 52
}

FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
    'SMA_20', 'EMA_20', 'Volatility'
]

CONFIG = {
    'symbols': SYMBOLS,
    'lookback_periods': LOOKBACK_PERIODS,
    'features': FEATURES,
    'hidden_size': 64,
    'num_layers': 2,
    'dropout': 0.2,
    'epochs': 20,
    'batch_size': 32,
    'lr': 0.001
}

print(f"🎯 Криптовалюты: {len(SYMBOLS)}")
print(f"⏱ Таймфреймы: {list(TIMEFRAMES_MAP.keys())}")
print(f"📊 Признаков: {len(FEATURES)}")

In [ ]:
# ==========================================
# 4. Функции загрузки и обработки данных
# ==========================================

def calculate_indicators(df):
    """Расчет технических индикаторов"""
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # Bollinger Bands
    sma = df['Close'].rolling(window=20).mean()
    std = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = sma + (std * 2)
    df['BB_Lower'] = sma - (std * 2)
    
    # SMA / EMA
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    
    # Volatility
    df['Volatility'] = df['Close'].pct_change().rolling(window=14).std()
    
    return df

def load_data(symbol, timeframe):
    """Загрузка данных через yfinance"""
    try:
        yf_tf = TIMEFRAMES_MAP.get(timeframe, '1d')
        # Загружаем больше данных для ресемплинга
        period = '1y' if timeframe in ['5m', '15m', '1h'] else '5y'
        
        df = yf.download(symbol, period=period, interval=yf_tf, progress=False)
        
        if df.empty:
            return None
            
        # Сброс мультииндекса если есть (новые версии yfinance)
        if isinstance(df.index, pd.MultiIndex):
            df = df.droplevel('Ticker', axis=0)
            
        df = df.dropna()
        
        # Ресемплинг для недостающих таймфреймов (4h, 12h)
        if timeframe == '4h':
            df = df.resample('4H').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna()
        elif timeframe == '12h':
            df = df.resample('12H').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
            }).dropna()
            
        if len(df) < 100:
            return None
            
        return df
        
    except Exception as e:
        print(f"❌ Ошибка загрузки {symbol} ({timeframe}): {e}")
        return None

print("✅ Функции данных определены")

In [ ]:
# ==========================================
# 5. Сбор данных и Feature Engineering
# ==========================================

all_data = {}

for symbol in SYMBOLS:
    print(f"\n📥 Обработка {symbol}...")
    symbol_data = {}
    
    for tf in TIMEFRAMES_MAP.keys():
        df = load_data(symbol, tf)
        if df is not None:
            df = calculate_indicators(df)
            df = df.dropna()
            # Убеждаемся, что есть все нужные колонки
            if all(col in df.columns for col in FEATURES):
                symbol_data[tf] = df
                print(f"  ✅ {tf}: {len(df)} строк")
            else:
                print(f"  ⚠️ {tf}: Недостаточно признаков")
        else:
            print(f"  ❌ {tf}: Нет данных")
    
    if symbol_data:
        all_data[symbol] = symbol_data
    else:
        print(f"⚠️ Пропускаем {symbol}, нет данных ни по одному ТФ")

if not all_data:
    raise ValueError("❌ Не удалось загрузить данные ни для одной криптовалюты!")

print(f"\n✅ Данные загружены для {len(all_data)} символов")

In [ ]:
# ==========================================
# 6. Подготовка данных для модели (Scaling & Sequences)
# ==========================================

def prepare_sequences(symbol_data_dict, lookback_periods, features):
    """
    Скалирует данные и создает последовательности для каждого ТФ.
    Возвращает словари: X_dict, y_dict, scalers_dict
    """
    X_dict = {}
    y_dict = {}
    scalers_dict = {}
    
    min_len = float('inf')
    
    # 1. Скалирование и определение общей длины
    for tf, df in symbol_data_dict.items():
        scaler = MinMaxScaler()
        # ВАЖНО: Берем только числовые колонки признаков, исключая Date если есть
        data_to_scale = df[features].copy()
        
        scaled_data = scaler.fit_transform(data_to_scale)
        scalers_dict[tf] = scaler
        
        lookback = lookback_periods.get(tf, 60)
        
        X_list = []
        y_list = []
        
        for i in range(lookback, len(scaled_data)):
            X_list.append(scaled_data[i-lookback:i])
            # Цель: рост цены (1) или падение (0)
            target = 1 if scaled_data[i, 3] > scaled_data[i-1, 3] else 0 # Index 3 = Close
            y_list.append(target)
            
        X_dict[tf] = np.array(X_list)
        y_dict[tf] = np.array(y_list)
        
        if len(X_list) < min_len:
            min_len = len(X_list)
            
    # 2. Выравнивание длин всех таймфреймов (берем минимум)
    # Чтобы можно было объединить их в один батч для multi-timeframe модели
    for tf in X_dict:
        X_dict[tf] = X_dict[tf][-min_len:]
        y_dict[tf] = y_dict[tf][-min_len:]
        
    return X_dict, y_dict, scalers_dict

print("✅ Функция подготовки данных готова")

In [ ]:
# ==========================================
# 7. Определение Модели (Multi-Timeframe LSTM)
# ==========================================

class MultiTimeframeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, num_timeframes):
        super(MultiTimeframeLSTM, self).__init__()
        
        self.num_timeframes = num_timeframes
        
        # Отдельный LSTM для каждого таймфрейма
        self.lstm_layers = nn.ModuleList([
            nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
            for _ in range(num_timeframes)
        ])
        
        # Attention механизм для объединения выходов
        self.attention = nn.Linear(hidden_size, 1)
        
        # Финальный классификатор
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x_list):
        # x_list: список тензоров [batch, seq_len, input_size] для каждого ТФ
        lstm_outputs = []
        
        for i, lstm in enumerate(self.lstm_layers):
            _, (hidden, _) = lstm(x_list[i])
            # Берем последний слой hidden state
            last_hidden = hidden[-1] 
            lstm_outputs.append(last_hidden)
            
        # Stack: [batch, num_timeframes, hidden_size]
        stacked = torch.stack(lstm_outputs, dim=1)
        
        # Attention weights
        attn_weights = torch.softmax(self.attention(stacked), dim=1)
        
        # Context vector
        context = torch.sum(attn_weights * stacked, dim=1)
        
        out = self.fc(context)
        return self.sigmoid(out)

print("✅ Модель определена")

In [ ]:
# ==========================================
# 8. Обучение моделей
# ==========================================

trained_models = {}

for symbol, tf_data in all_data.items():
    print(f"\n{'='*60}")
    print(f"🚀 Обучение: {symbol}")
    print(f"{'='*60}")
    
    # Подготовка данных
    X_dict, y_dict, scalers = prepare_sequences(
        tf_data, 
        CONFIG['lookback_periods'], 
        CONFIG['features']
    )
    
    num_tf = len(X_dict)
    if num_tf == 0:
        print("⚠️ Нет данных для обучения после предобработки")
        continue
        
    # Конвертация в Tensor
    X_tensors = [torch.FloatTensor(X_dict[tf]).to(device) for tf in X_dict]
    y_tensor = torch.FloatTensor(y_dict[list(X_dict.keys())[0]]).unsqueeze(1).to(device)
    
    # Разделение на Train/Test (80/20)
    split_idx = int(len(X_tensors[0]) * 0.8)
    
    X_train = [x[:split_idx] for x in X_tensors]
    X_test = [x[split_idx:] for x in X_tensors]
    y_train = y_tensor[:split_idx]
    y_test = y_tensor[split_idx:]
    
    print(f"📦 Train: {split_idx}, Test: {len(y_tensor) - split_idx}")
    
    # Инициализация модели
    input_size = len(CONFIG['features'])
    model = MultiTimeframeLSTM(
        input_size=input_size,
        hidden_size=CONFIG['hidden_size'],
        num_layers=CONFIG['num_layers'],
        dropout=CONFIG['dropout'],
        num_timeframes=num_tf
    ).to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    
    best_loss = float('inf')
    
    # Цикл обучения
    for epoch in range(CONFIG['epochs']):
        model.train()
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Validation каждые 5 эпох
        if (epoch + 1) % 5 == 0 or epoch == CONFIG['epochs'] - 1:
            model.eval()
            with torch.no_grad():
                test_pred = model(X_test)
                test_loss = criterion(test_pred, y_test)
                acc = ((test_pred > 0.5).float() == y_test).float().mean()
                
            print(f"Epoch [{epoch+1}/{CONFIG['epochs']}] Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Acc: {acc.item():.2%}")
            
            if test_loss < best_loss:
                best_loss = test_loss
                torch.save(model.state_dict(), f"/kaggle/working/models/{symbol}_best.pth")
    
    trained_models[symbol] = model
    print(f"💾 Модель сохранена: /kaggle/working/models/{symbol}_best.pth")

print("\n✅ Обучение всех моделей завершено!")

In [ ]:
# ==========================================
# 9. Сохранение результатов и метаданных
# ==========================================

import json

results = {
    "status": "completed",
    "symbols_trained": list(trained_models.keys()),
    "config": CONFIG,
    "device": str(device),
    "timestamp": datetime.now().isoformat()
}

with open('/kaggle/working/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("📄 Результаты сохранены в training_results.json")
print("📂 Проверьте папку '/kaggle/working/models' для весов моделей.")
print("\n🎉 Готово! Теперь вы можете скачать модели или использовать их для инференса.")